<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 5 · Inteligencia de tiempo
**Analista de datos · Power BI + Tableau** — Acumulado del año, año anterior y variación · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/05_inteligencia_de_tiempo.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-05)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Comparar el acumulado de 2025 con el de 2024 para cualquier sede.
2. Ver la variación mensual de cada sede frente al año anterior.
3. Tener la tabla de control de las medidas de tiempo.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Preparar ventas, metas y el Calendario
validas_t = validas.copy()
metas_c = metas.merge(sedes[["id_sede", "ciudad"]], on="id_sede")
mensual = validas_t.groupby(["ciudad", "anio", "mes"])["subtotal"].sum()
def serie(sede, anio):
    s = validas_t[validas_t.anio == anio]
    if sede != "Todas": s = s[s.ciudad == sede]
    return s.groupby("mes")["subtotal"].sum().reindex(range(1, 13), fill_value=0)
display(Markdown("Listo: venta neta mensual por sede y año (sin anuladas)."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo va el año frente al anterior? |
> | **Qué se grafica** | Líneas: venta acumulada de 2025 (TOTALYTD) y de 2024 (SAMEPERIODLASTYEAR), mes a mes |
> | **Por qué esta forma** | El acumulado muestra la distancia entre los años en cada momento del año. |
> | **Qué se descartó** | Barras mensuales: un mes malo parece una tendencia. |
> | **Cómo se lee** | La distancia vertical entre las líneas es la diferencia acumulada. Si se abre, el año va mejor que el anterior. |
> | **En Excel 365** | Venta YTD = TOTALYTD([Venta neta], Calendario[Date]); Venta YTD AA = CALCULATE([Venta YTD], SAMEPERIODLASTYEAR(Calendario[Date])). |

In [ ]:
# @title Gráfico 1 · Acumulado del año
sede = "Todas"  # @param ["Todas", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
y24, y25 = serie(sede, 2024).cumsum() / 1e6, serie(sede, 2025).cumsum() / 1e6
x = pd.to_datetime([f"2025-{m:02d}-01" for m in range(1, 13)])
fig = go.Figure()
fig.add_scatter(x=x, y=y25, name="Venta YTD 2025", line=dict(color=AZUL, width=3))
fig.add_scatter(x=x, y=y24, name="Venta YTD AA (2024)", line=dict(color=GRIS, width=2, dash="dash"))
var = y25.iloc[-1] / y24.iloc[-1] - 1
eje_meses(fig, x)
fig.update_layout(title=f"{sede if sede != 'Todas' else 'La empresa'} cerró 2025 con {porc(var, signo=True)} frente a 2024",
                  yaxis_title=MM, height=380, legend=dict(orientation="h", y=-0.2))
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿En qué meses creció o cayó cada sede? |
> | **Qué se grafica** | Mapa de calor: variación % contra el mismo mes del año anterior, por sede y mes |
> | **Por qué esta forma** | Seis sedes por doce meses: la cuadrícula muestra patrones (una sede débil todo el año o un mes malo para todas). |
> | **Qué se descartó** | Seis gráficos de líneas: obliga a comparar entre gráficos. |
> | **Cómo se lee** | Azul es crecimiento, magenta es caída. Una fila con mucho magenta es una sede con problemas, no un mes malo. |
> | **En Excel 365** | Matriz con Sedes[ciudad] en filas, Calendario[Mes] en columnas y [Var % AA] con formato condicional. |

In [ ]:
# @title Gráfico 2 · Variación mensual por sede
z = pd.DataFrame({c: serie(c, 2025) / serie(c, 2024) - 1 for c in ORDEN_CIUDADES}).T
fig = go.Figure(go.Heatmap(z=z.values, x=MESES, y=z.index, zmid=0, colorscale=[[0, MAGENTA], [0.5, "#F4F6FB"], [1, AZUL]],
                           text=[[porc(v, signo=True) for v in fila] for fila in z.values], texttemplate="%{text}", colorbar=dict(tickformat=".0%")))
negativos = (z < 0).sum(axis=1)
fig.update_layout(title=f"{negativos.idxmax()} tuvo más meses por debajo del año anterior ({negativos.max()} de 12)",
                  yaxis=dict(autorange="reversed"), height=380)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué sede creció más en el año? |
> | **Qué se grafica** | Barras: variación 2025 frente a 2024 por sede · Línea: variación de la empresa |
> | **Por qué esta forma** | Es la tabla de control de Var % AA en forma de ranking. |
> | **Qué se descartó** | Solo la cifra total: no se ve quién empuja y quién frena. |
> | **Cómo se lee** | Las sedes por encima de la línea crecen más que la empresa. |
> | **En Excel 365** | Matriz con [Venta neta], [Venta AA] y [Var % AA] por sede, filtro Año = 2025. |

In [ ]:
# @title Gráfico 3 · Variación anual por sede
v = pd.Series({c: serie(c, 2025).sum() / serie(c, 2024).sum() - 1 for c in ORDEN_CIUDADES}).sort_values()
total = serie("Todas", 2025).sum() / serie("Todas", 2024).sum() - 1
fig = go.Figure(go.Bar(x=v.values, y=v.index, orientation="h", marker_color=[AZUL if x > total else GRIS for x in v.values],
                       text=[porc(x, signo=True) for x in v.values], textposition="outside", cliponaxis=False))
fig.add_vline(x=total, line_dash="dash", line_color=MAGENTA, annotation_text=f"Empresa {porc(total, signo=True)}")
fig.update_layout(title=f"{v.index[-1]} creció {porc(v.iloc[-1], signo=True)}; la empresa, {porc(total, signo=True)}",
                  xaxis_tickformat=".0%", height=360, xaxis_range=[0, v.max() * 1.25])
fig.show()

## Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Cuánto creció la empresa en 2025 frente a 2024?
respuesta = "Elige una opción"  # @param ["Elige una opción", "+12,0 %", "+18,9 %", "+25,2 %", "+39,1 %"]
revisar(respuesta, porc(serie("Todas", 2025).sum() / serie("Todas", 2024).sum() - 1, signo=True), """Es el final de la línea azul del gráfico 1 con «Todas».""")

In [ ]:
# @title Reto 2 · ¿Qué sede creció más en 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
revisar(respuesta, pd.Series({c: serie(c, 2025).sum() / serie(c, 2024).sum() - 1 for c in ORDEN_CIUDADES}).idxmax(), """Es la barra más larga del gráfico 3.""")

In [ ]:
# @title Reto 3 · Venta AA sale vacía en todas las filas. ¿Qué revisas primero?
respuesta = "Elige una opción"  # @param ["Elige una opción", "El color del visual", "Que la medida use Calendario[Date] y que el Calendario esté marcado y relacionado", "El número de decimales", "La segmentación de canal"]
revisar(respuesta, "Que la medida use Calendario[Date] y que el Calendario esté marcado y relacionado", """Es la condición para que funcionen las funciones de tiempo.""")

## Apóyate en IA
Para crear medidas de tiempo:

```text
En Power BI tengo [Venta neta] y un Calendario marcado (columna Date).
Escribe Venta YTD, Venta AA y Var % AA, y dame una prueba con valores que pueda comparar.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* TOTALYTD acumula; SAMEPERIODLASTYEAR desplaza un año; DIVIDE protege la variación.
* Todo depende de un Calendario marcado y relacionado.
* Una misma medida responde para cualquier sede, canal o mes.

**Siguiente:** [Clave 6 · Informe que responde preguntas](https://omarli-360.github.io/powerbi-tableau-analista/#clave-06) · [Cuaderno 06](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/06_informe_que_responde.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>